In [2]:
# 一、准备文档
from langchain_text_splitters import MarkdownHeaderTextSplitter
from dotenv import load_dotenv
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings

# 加载配置文件
load_dotenv()
# 创建向量模型
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key="sk-a434e0ce54c74556aea762c4ded941b0"
)
# 初始化Collection
vectorstore = Milvus(
    embeddings,                             # 稠密向量模型
    collection_name="langchain_collection", # collection名称
    builtin_function=BM25BuiltInFunction(   # 生成稀疏向量的函数
        analyzer_params={"type": "chinese"} # 指定中文分词
    ),
    # 【注意】：列表的第一个为稠密向量，第二个为稀疏向量
    vector_field=["dense", "sparse"],       # 向量字段，包括稠密和稀疏，
    connection_args={
        "uri": settings.rag.milvus_url,     # milvus的uri路径
    },
    # index_params=[index_param],             # 可选，索引参数，必须与vector_field对应
    drop_old=True,                          # 是否删除旧的collection，避免重复创建
)
print("> 向量库信息已配置完毕")


def prepare_docs():
    # =============1.准备文档================
    with open('./resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    # 为了方便查看，给每个Document加一个id，从1开始递增
    # 也可以让 milvus 自动生成
    # - LangChain写入Milvus的文档如果自定义ID，则ID必须是字符串类型！
    # - 也可以不指定ID，则ID由Milvus生成，为INT64类型
    #for i, doc in enumerate(docs, start=1):
        #doc.id = str(i)
    return docs

docs = prepare_docs()
# 四、LangChain添加数据
res = vectorstore.add_documents(docs)
print(f"> 添加文档数据成功：{res}")


> 向量库信息已配置完毕
> 添加文档数据成功：[470200415637511257, 470200415637511258, 470200415637511259, 470200415637511260, 470200415637511261, 470200415637511262, 470200415637511263, 470200415637511264, 470200415637511265, 470200415637511266, 470200415637511267, 470200415637511268, 470200415637511269, 470200415637511270, 470200415637511271, 470200415637511272, 470200415637511273, 470200415637511274, 470200415637511275, 470200415637511276, 470200415637511277]
